In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

# U-Net — 对称编解码器 + 跳跃连接

## 和 FCN 的关系

FCN 你已经写过了。U-Net = FCN 的进化版，两个关键改进：

| | FCN | U-Net |
|---|---|---|
| 上采样路径 | 简单的双线性插值或转置卷积 | 对称的多层 Decoder |
| 浅层特征 | 丢失（下采样后回不去） | 通过 **跳跃连接** 拼回 Decoder |
| 输出质量 | 粗糙，边界模糊 | 精细，边界保留 |

## 数据流

```
输入 (B, C, H, W)
  ↓
[Encoder — 逐层下采样，提取语义]
  Conv-BN-ReLU ×2 → MaxPool   → C×1, H/2, W/2   (enc1)
  Conv-BN-ReLU ×2 → MaxPool   → C×2, H/4, W/4   (enc2)
  Conv-BN-ReLU ×2 → MaxPool   → C×4, H/8, W/8   (enc3)
  Conv-BN-ReLU ×2 → MaxPool   → C×8, H/16, W/16 (enc4)
  ↓
[Bottleneck — 最深语义信息]
  Conv-BN-ReLU ×2             → C×16, H/16, W/16
  ↓
[Decoder — 逐层上采样 + 跳跃连接，恢复空间细节]
  Up → concat enc4 → Conv×2  → C×8, H/8, W/8
  Up → concat enc3 → Conv×2  → C×4, H/4, W/4
  Up → concat enc2 → Conv×2  → C×2, H/2, W/2
  Up → concat enc1 → Conv×2  → C×1, H,   W
  ↓
1×1 Conv → (B, n_classes, H, W)
```

**核心思想**：Encoder 告诉你"这是什么"，Decoder 告诉你"这在哪"，跳过连接把两者粘在一起。

## 1. 基础卷积块 — DoubleConv

U-Net 每个阶段都是 **两个 3×3 Conv + BN + ReLU** 的组合，不是单层卷积。

```
Conv3x3 → BN → ReLU → Conv3x3 → BN → ReLU
```

为什么是两层而不是一层？单层感受野太小，两层 3×3 的感受野等价于一层 5×5，但参数更少。

In [ ]:
class DoubleConv(nn.Module):
    """两次 3×3 卷积 + BN + ReLU，U-Net 的基本构建块"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            # 第一层：改变通道数（in → out），可选
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            # 第二层：保持通道数不变（out → out），加深非线性
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)

## 2. 下采样模块 — Encoder 的每一级

```
DoubleConv → MaxPool2d(2)
```

MaxPool 把空间尺寸减半，通道数翻倍（通过 DoubleConv 控制）。每个 encoder 阶段输出两份：一份给下一层，一份存下来等跳跃连接用。

In [ ]:
class Down(nn.Module):
    """Encoder 的一级：DoubleConv → MaxPool2d 下采样"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = DoubleConv(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        before_pool = self.conv(x)  # 存下来给跳跃连接用
        after_pool = self.pool(before_pool)  # 传给下一层
        return before_pool, after_pool

## 3. 上采样模块 — Decoder 的每一级

```
转置卷积(×2 上采样) → concat(跳跃连接) → DoubleConv
```

这是 U-Net 的核心操作：
1. 用转置卷积把特征图放大一倍
2. 把 Encoder 对应层的特征拼接到通道维上（**这就是跳跃连接**）
3. 再用 DoubleConv 融合两路特征

注意：如果 Encoder 和 Decoder 的特征图尺寸不匹配（奇数输入导致），需要裁剪 Encoder 特征。

In [ ]:
class Up(nn.Module):
    """Decoder 的一级：转置卷积上采样 → concat Encoder 特征 → DoubleConv"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # 转置卷积：把空间尺寸 ×2，同时把通道数从 in_channels 压到 out_channels
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)

        # concat 后通道数加倍（一半来自 skip，一半来自上采样）
        # 所以 DoubleConv 的输入是 out_channels × 2
        self.conv = DoubleConv(out_channels * 2, out_channels)

    def forward(self, x_decoder, x_encoder):
        """
        x_decoder: 来自上一层 Decoder（或 Bottleneck），低分辨率高语义
        x_encoder: 来自对应 Encoder 层，高分辨率低语义（跳跃连接）
        """
        # 1. 上采样：把低分辨率特征图放大到和高分辨率一样
        x_up = self.up(x_decoder)

        # 2. 处理尺寸不匹配（输入尺寸不是 2^N 倍时）
        diff_h = x_encoder.size(2) - x_up.size(2)
        diff_w = x_encoder.size(3) - x_up.size(3)
        # 四周各裁一半，让两个特征图尺寸完全对齐
        x_up = F.pad(x_up, [
            diff_w // 2, diff_w - diff_w // 2,   # 左右
            diff_h // 2, diff_h - diff_h // 2,   # 上下
        ])

        # 3. 跳跃连接：把 Encoder 的高分辨率特征拼到通道维上
        x = torch.cat([x_encoder, x_up], dim=1)   # 通道数翻倍

        # 4. DoubleConv 融合两路特征
        return self.conv(x)

## 4. U-Net 完整模型

四大组件拼接：
- **Encoder**：4 级 Down 模块，逐层语义抽象
- **Bottleneck**：最深层，全局语义
- **Decoder**：4 级 Up 模块，逐层恢复空间（+跳跃连接）
- **Head**：1×1 卷积，把特征图映射到类别数

默认`features=[64, 128, 256, 512]`是原论文配置。可以根据任务复杂度缩放（小任务 16/32/64/128，大任务 64/128/256/512）。

In [ ]:
class UNet(nn.Module):
    """U-Net：对称编解码 + 跳跃连接的语义分割网络

    Args:
        in_channels:  输入图像通道数（RGB=3，灰度=1）
        n_classes:    分割类别数（含背景，如 Cityscapes 有 19 类前景 + 1 背景 = 20）
        features:     每层特征通道数列表，从浅到深
    """
    def __init__(self, in_channels=3, n_classes=1, features=[64, 128, 256, 512]):
        super().__init__()
        self.in_channels = in_channels
        self.n_classes = n_classes

        # ========================
        # Encoder（下采样路径）
        # ========================
        # 每一级：DoubleConv → MaxPool，通道数翻倍，空间减半
        self.enc1 = Down(in_channels, features[0])    # → feat[0], H/2
        self.enc2 = Down(features[0], features[1])    # → feat[1], H/4
        self.enc3 = Down(features[1], features[2])    # → feat[2], H/8
        self.enc4 = Down(features[2], features[3])    # → feat[3], H/16

        # ========================
        # Bottleneck（最深层）
        # ========================
        # 不下采样，只做 DoubleConv，通道数继续翻倍
        self.bottleneck = DoubleConv(features[3], features[3] * 2)
        # → feat[3]×2, H/16

        # ========================
        # Decoder（上采样路径）
        # ========================
        # 每一级：转置卷积(x2) → concat(Encoder 特征) → DoubleConv
        # in_channels = 上一层输出的通道数
        self.up4 = Up(features[3] * 2, features[3])   # → feat[3], H/8
        self.up3 = Up(features[3], features[2])        # → feat[2], H/4
        self.up2 = Up(features[2], features[1])        # → feat[1], H/2
        self.up1 = Up(features[1], features[0])        # → feat[0], H

        # ========================
        # Head（输出层）
        # ========================
        # 1×1 卷积：把多通道特征图映射到类别得分
        # kernel_size=1 意味着逐像素独立分类，不改变空间尺寸
        self.head = nn.Conv2d(features[0], n_classes, kernel_size=1)

    def forward(self, x):
        # ---------- Encoder ----------
        enc1, x = self.enc1(x)   # enc1 存下来等 Up1 用
        enc2, x = self.enc2(x)   # enc2 存下来等 Up2 用
        enc3, x = self.enc3(x)   # enc3 存下来等 Up3 用
        enc4, x = self.enc4(x)   # enc4 存下来等 Up4 用

        # ---------- Bottleneck ----------
        x = self.bottleneck(x)

        # ---------- Decoder（跳跃连接）----------
        x = self.up4(x, enc4)    # 上采样 + concat enc4
        x = self.up3(x, enc3)    # 上采样 + concat enc3
        x = self.up2(x, enc2)    # 上采样 + concat enc2
        x = self.up1(x, enc1)    # 上采样 + concat enc1

        # ---------- Head ----------
        return self.head(x)      # (B, n_classes, H, W)